# WAV2HYP catalog explorer

Set **`CONFIG_PATH`** (and optional path overrides) in **Setup**, then run the **Catalog** cell to load `nll.h5`. Adjust **`VIEW_DATE`** and **`EVENT_ID`** in the later sections.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Image, display

from wav2hyp.utils.io import NLLOutput


## Setup

Run the next two cells after editing paths.


In [ ]:
# Path to your WAV2HYP YAML (inventory + output layout).
# Works whether the notebook's working directory is the repo root or notebooks/.
_cwd = Path.cwd()
if (_cwd / "examples/sthelens.yaml").is_file():
    CONFIG_PATH = (_cwd / "examples/sthelens.yaml").resolve()
elif (_cwd.parent / "examples/sthelens.yaml").is_file():
    CONFIG_PATH = (_cwd.parent / "examples/sthelens.yaml").resolve()
else:
    CONFIG_PATH = (_cwd / "examples/sthelens.yaml").resolve()

# Pipeline output root: must contain locations/nll.h5 unless you set LOCATOR_H5 below.
# If None, uses output.base_dir from the YAML (relative to the YAML file's directory).
RUN_BASE_DIR = None

# Folder with optional pre-rendered figures (catalog map PNG, clipboards/). Defaults to RUN_BASE_DIR.
ARTIFACTS_DIR = None

# Set to an explicit nll.h5 path if the catalog file is not under RUN_BASE_DIR/locations/.
LOCATOR_H5 = None

# Pre-rendered catalog map filename inside ARTIFACTS_DIR (optional).
CATALOG_MAP_PNG_NAME = "sthelens_map.png"
CLIPBOARDS_SUBDIR = "clipboards"


In [ ]:
def _resolve(cfg_path: Path, rel_or_abs: str) -> Path:
    p = Path(rel_or_abs)
    return p.resolve() if p.is_absolute() else (cfg_path.parent / p).resolve()


with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)

RUN_BASE_DIR = Path(RUN_BASE_DIR).resolve() if RUN_BASE_DIR is not None else _resolve(
    CONFIG_PATH, _cfg["output"]["base_dir"]
)
ARTIFACTS_DIR = Path(ARTIFACTS_DIR).resolve() if ARTIFACTS_DIR is not None else RUN_BASE_DIR
loc_sub = _cfg["output"].get("locator_dir", "locations")
LOCATOR_H5 = Path(LOCATOR_H5).resolve() if LOCATOR_H5 is not None else (RUN_BASE_DIR / loc_sub / "nll.h5")
INVENTORY_PATH = _resolve(CONFIG_PATH, _cfg["inventory"]["file"])

nll = NLLOutput(str(LOCATOR_H5))
catalog_df = nll.read_catalog_table()

print("CONFIG_PATH     ", CONFIG_PATH)
print("RUN_BASE_DIR    ", RUN_BASE_DIR)
print("ARTIFACTS_DIR   ", ARTIFACTS_DIR)
print("INVENTORY_PATH  ", INVENTORY_PATH)
print("LOCATOR_H5      ", LOCATOR_H5, "exists:", LOCATOR_H5.exists())
print("catalog rows    ", len(catalog_df))


## Catalog

Histogram (events per UTC day) and map preview from `catalog_df`. If you saved a pipeline map image, it is shown when present.


In [ ]:
cat = catalog_df.copy()
cat["origin_time"] = pd.to_datetime(cat["origin_time"], utc=True, errors="coerce")
cat["utc_date"] = cat["origin_time"].dt.floor("D")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cat["utc_date"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Events per UTC day")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_xlabel("Date")

axes[1].scatter(cat["longitude"], cat["latitude"], s=8, alpha=0.6, c="tab:blue")
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].set_title("Catalog map (quick scatter)")
plt.tight_layout()
plt.show()

_map = ARTIFACTS_DIR / CATALOG_MAP_PNG_NAME
if _map.is_file():
    display(Image(filename=str(_map)))
else:
    print(f"No map PNG at {_map} (optional)")


## Day view

Pick a **UTC calendar date** that overlaps your origins.


In [ ]:
VIEW_DATE = "2004-09-23"


In [ ]:
day_start = pd.Timestamp(VIEW_DATE, tz="UTC")
day_end = day_start + pd.Timedelta(days=1)
day_df = catalog_df[
    (pd.to_datetime(catalog_df["origin_time"], utc=True) >= day_start)
    & (pd.to_datetime(catalog_df["origin_time"], utc=True) < day_end)
].copy()

display(day_df)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(day_df["longitude"], day_df["latitude"], s=20, alpha=0.85)
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Origins on {VIEW_DATE} (UTC)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


## Event view

Set **`EVENT_ID`** to the full `event_id` string from `catalog_table` (copy from the table above). Optional: show arrivals and any matching images under `ARTIFACTS_DIR / CLIPBOARDS_SUBDIR`.


In [ ]:
EVENT_ID = "paste-full-event_id-here"
# If clipboard PNG names do not include the full EVENT_ID, set a substring that appears in the filename (e.g. short hash).
CLIPBOARD_NAME_SUBSTRING = None


In [ ]:
ev_df = catalog_df.loc[catalog_df["event_id"].astype(str) == str(EVENT_ID)]
display(ev_df)

if len(ev_df) == 1:
    arr = nll.read_arrivals(event_ids=[EVENT_ID])
    display(arr)
else:
    print("No single match — check EVENT_ID against the catalog table.")

_clip = ARTIFACTS_DIR / CLIPBOARDS_SUBDIR
_placeholder = "paste-full-event_id-here"
_needle = CLIPBOARD_NAME_SUBSTRING or (None if EVENT_ID == _placeholder else EVENT_ID)
if _clip.is_dir() and _needle:
    hits = sorted(p for p in _clip.iterdir() if _needle in p.name)
    for p in hits:
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".gif", ".webp"}:
            display(Image(filename=str(p)))
    if not hits:
        print(f"No clipboard files under {_clip} with {_needle!r} in the name.")
elif _clip.is_dir():
    print("Set EVENT_ID (and CLIPBOARD_NAME_SUBSTRING if needed) to search clipboards.")
else:
    print(f"No clipboards directory at {_clip} (optional)")
